In [21]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [22]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")


'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n2. Open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\nTo test that the local server is running, you can also run:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you get a `Connection refused` error while prompting Ollama during the homework, restart the Ollama server with:\n\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [9]:
#assistant.rag("How do I run Olama locally?")
search("How do I run Olama locally?")

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': 'fe8fed31e6',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'How do I get token counts for Module 1 homework if I use a different provider?',
  'answer': "For the current Module 1 homework, get the token

In [2]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes, in most cases you can join if enrollment is still open.\n\nA few quick things to check:\n- **Enrollment deadline**: Is the course still accepting students?\n- **Prerequisites**: Do you meet any required background or prior courses?\n- **Capacity**: Is there still room?\n- **Registration process**: Do you need to sign up through a portal or contact the instructor/admin?\n\nIf you want, I can help you figure out what to ask or draft a message like:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is enrollment still open, and if so, what do I need to do to register?\n\nIf you share the course name or where it’s offered, I can help you check what steps usually apply.'

In [15]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [16]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join"}', call_id='call_zij3DCw77cHvJaKzHx0SN4iB', name='search', type='function_call', id='fc_0f814623b0782201006a3eee2aaa1c819a8f816d65db589e4e', namespace=None, status='completed')]

In [22]:
import json
len(response.output)
call = response.output[0]
print(call.arguments)
print(type(call.arguments))
test = json.loads(call.arguments)
print(test)

{"query":"join course discovered late enrollment can I join"}
<class 'str'>
{'query': 'join course discovered late enrollment can I join'}


In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you 

In [12]:
messages.extend(response.output)
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [15]:
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I still join"}', call_id='call_itliCzlDiG7EE6TSERhB7apf', name='search', type='function_call', id='fc_0f1f1f627788d429006a3bcfd2021c8199802ea0cfa33946c4', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_itliCzlDiG7EE6TSERhB7apf', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you ne

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(810, 29)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(810, 29)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001389


In [32]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search funtion.
Use as many keywords from the user question as possible when making first request.

Make multimple searches.

Try to expand your search by using new key words based on the results you get from the search.

At the end ask  if there are othe areas that the user wants to explore.
"""

In [39]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        results = search(**args)

    result_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id" : call.call_id,
        "output" : result_json
    }

In [34]:
question = "I just discovered the course. Can I join it?"

In [35]:
messages = [
    {'role': 'developer','content': instructions},
    {'role': 'user', 'content': question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)


In [36]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join FAQ"}', call_id='call_VmZsdh4K9Gq7IP3vuOQMjsJq', name='search', type='function_call', id='fc_016bf9a3094cbdaf006a4039da8dd8819ab94ca2889273f272', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I join course access enrollment FAQ"}', call_id='call_c0vO277XCbC81zMQ4itJYUOM', name='search', type='function_call', id='fc_016bf9a3094cbdaf006a4039da8dec819aadb5b9039047c9cc', namespace=None, status='completed')]

In [37]:
messages.extend(response.output)
has_function_calls = False

In [40]:
for item in response.output:
    if item.type == "function_call":
        print("function call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
function call: search {"query":"new student discovered course can I join course access enrollment FAQ"}


In [41]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and doing homework right away; registration isn’t required to begin.

If you’d like, I can also help you with how to start the course or explain the certificate requirements. Are there other areas you want to explore?


In [42]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [43]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama local installation model serve start FAQ"}
function_call: search {"query":"Ollama local run install macOS Windows Linux FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, first install it for your OS:

- **macOS**: download the `.pkg` from https://ollama.com/download and install it
- **Windows**: download the `.msi` from https://ollama.com/download and install it
- **Linux**:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Then start a model locally:

```bash
ollama run llama3
```

This will download the model, start it locally, and open a chat-like terminal session.

To check that the local server is running, use:

```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client:

```bash
pip install ollama
```

Example:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": "Hello!"}]
)

print(respons

'To run **Ollama locally**, first install it for your OS:\n\n- **macOS**: download the `.pkg` from https://ollama.com/download and install it\n- **Windows**: download the `.msi` from https://ollama.com/download and install it\n- **Linux**:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nThen start a model locally:\n\n```bash\nollama run llama3\n```\n\nThis will download the model, start it locally, and open a chat-like terminal session.\n\nTo check that the local server is running, use:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client:\n\n```bash\npip install ollama\n```\n\nExample:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\nIf you get a connection issue, restarting the server with:\n\n```bash\nollama serve\n```\n\nor in notebook environments:\n\n```bash\n!nohup ollama 

In [44]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course still join enroll late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. You can start learning and following the materials whenever you want.

If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.

Also, you don’t need a special confirmation to begin; you can just start with the course materials and submit homework while the forms are open.

If you’d like, I can also help you with:
- how to start the course,
- the weekly workflow,
- or certificate requirements.

Are there other areas you want to explore?


'Yes — you can still join the course. You can start learning and following the materials whenever you want.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still being accepted.\n\nAlso, you don’t need a special confirmation to begin; you can just start with the course materials and submit homework while the forms are open.\n\nIf you’d like, I can also help you with:\n- how to start the course,\n- the weekly workflow,\n- or certificate requirements.\n\nAre there other areas you want to explore?'

In [45]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enroll discovered course can I join FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while still accepting submissions peer review live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open and complete the required peer reviews during the live cohort.

If you’d like, I can also help you with the next steps for joining or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open and complete the required peer reviews during the live cohort.\n\nIf you’d like, I can also help you with the next steps for joining or explain the certificate requirements.'

In [46]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit opening chess FAQ"}
function_call: search {"query":"what is the queen's gambit chess opening"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn as a gambit to try to **gain control of the center** and put pressure on Black’s d5 pawn.

A quick idea of the opening:
- **White** is aiming for more space and central influence.
- **Black** can accept or decline the pawn sacrifice.
- It’s one of the most famous and important openings in chess.

If you want, I can also explain:
- **why it’s called a gambit**
- **the main variations**
- or **how to play it as White or defend against it as Black**.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn as a gambit to try to **gain control of the center** and put pressure on Black’s d5 pawn.\n\nA quick idea of the opening:\n- **White** is aiming for more space and central influence.\n- **Black** can accept or decline the pawn sacrifice.\n- It’s one of the most famous and important openings in chess.\n\nIf you want, I can also explain:\n- **why it’s called a gambit**\n- **the main variations**\n- or **how to play it as White or defend against it as Black**.'

In [47]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit course FAQ chess queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit FAQ course"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it may be off-topic for this course.

If you meant something course-related, feel free to rephrase and I can look it up. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it may be off-topic for this course.\n\nIf you meant something course-related, feel free to rephrase and I can look it up. Are there other areas you want to explore?'